# 16 — USA Central Valley, USGS NAIP Plus with Delineate-Anything

Delineates large commercial agricultural fields in California's Central Valley using **USGS NAIP Plus** imagery — the same NAIP data available on GEE but acquired directly from the [USGS USGSNAIPPlus ImageServer](https://imagery.nationalmap.gov/arcgis/rest/services/USGSNAIPPlus/ImageServer) — through the `usgs-naip-plus` source and the **Delineate-Anything** (YOLO) engine.

**Estimated runtime:** ~30–60 minutes (1 year, high-resolution imagery, GPU recommended).

**Prerequisites:**
```bash
pip install "agribound[delineate-anything]"
```

**Notes:**
- No Google Earth Engine authentication is required.
- This example uses the non-GEE `usgs-naip-plus` source.
- `lulc_filter=False` is set intentionally to keep the workflow purely non-GEE.
- If the standalone Delineate-Anything repository is not present locally, agribound falls back to direct YOLO inference.

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound

OUTPUT_DIR = Path("../../outputs/usa_central_valley_usgs_naip_plus")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = "usgs-naip-plus"
ENGINE = "delineate-anything"
YEAR = 2022
USGS_STATE = "CA"

## Create Study Area

AOI near Fresno, CA — large commercial agriculture.

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-119.85, 36.70],
                        [-119.75, 36.70],
                        [-119.75, 36.78],
                        [-119.85, 36.78],
                        [-119.85, 36.70],
                    ]
                ],
            },
            "properties": {"name": "Central Valley AOI"},
        }
    ],
}

study_area = str(OUTPUT_DIR / "central_valley_aoi.geojson")
with open(study_area, "w", encoding="utf-8") as f:
    json.dump(aoi, f, indent=2)

print(f"Study area: {study_area}")

## Run Delineation

In [ ]:
output_path = OUTPUT_DIR / f"fields_usgs_naip_plus_{YEAR}.gpkg"

gdf = agribound.delineate(
    study_area=study_area,
    source=SOURCE,
    year=YEAR,
    engine=ENGINE,
    output_path=str(output_path),
    usgs_state=USGS_STATE,
    lulc_filter=False,
    min_field_area_m2=10000,
    simplify_tolerance=3.0,
    n_workers=8,
)

print(f"Delineated {len(gdf)} fields")
if "metrics:area" in gdf.columns and len(gdf) > 0:
    area_ha = gdf["metrics:area"].sum() / 10000
    print(f"Total agricultural area: {area_ha:,.1f} ha")
    print(f"Average field size: {gdf['metrics:area'].mean() / 10000:,.1f} ha")

## Visualization

In [ ]:
m = agribound.show_boundaries(
    gdf,
    basemap="Google.Satellite",
    output_html=str(OUTPUT_DIR / "map_central_valley_usgs_naip_plus.html"),
)
m